In [0]:
# Imports
import csv
from pathlib import Path

from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

## Declaração de constantes

In [0]:
CATALOGO     = "ANP_Combustiveis"

schemaRaw    = "00_raw"
schemaBronze = "01_bronze"
Volume       = "data"

volumePath   = Path(f"/Volumes/{CATALOGO}/{schemaRaw}/{Volume}")

paths = {"PRECOS_CSV":     Path(volumePath / "ANP" / "PRECOS" / "CSV"),
         "VENDAS":         Path(volumePath / "ANP" / "VENDAS"),
         "IBGE":           Path(volumePath / "IBGE"),
         "PRECOS_PARQUET": Path(volumePath / "PARQUET" / "PRECOS_REVENDA")
         }

tables = {"PRECOS_REVENDA":   f"`{CATALOGO}`.`{schemaBronze}`.`precos_revenda`",
          "VENDAS_MUNICIPIO": f"`{CATALOGO}`.`{schemaBronze}`.`vendas_municipio`",
          "MUNICIPIOS_IBGE":  f"`{CATALOGO}`.`{schemaBronze}`.`municipios_ibge`",
          "ESTADOS_IBGE":     f"`{CATALOGO}`.`{schemaBronze}`.`estados_ibge`"}

##### Declarar função utilizada para atualizar tabela Delta

In [0]:
# Função auxiliar para carregar dados na tabela Delta
def mergeToDelta(df: DataFrame, tableName: str, condition: str):
    deltaTable = DeltaTable.forName(spark, tableName)

    (
        deltaTable.alias("target")
        .merge(
            df.alias("source"),
            condition
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"[Merge-DeltaTable] Carga concluída em: {tableName}")

##### Carga de Precos de revenda

In [0]:
# Leitura dos arquivos .csv da ANP (Lê todos .csv do diretorio simultaneamente)
precosRawDf = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(str(paths["PRECOS_CSV"] / "*.csv"))
)

# Seleciona e renomeia as colunas a fim de compatibilizar o schema da tabela Bronze 'precos_revenda'
precosDf = (
    precosRawDf
    .select(
        F.col("Regiao - Sigla").alias("regiao"),
        F.col("Estado - Sigla").alias("uf"),
        F.col("Municipio").alias("municipio"),
        F.col("Revenda").alias("razao_social"),
        F.col("CNPJ da Revenda").alias("cnpj_revenda"),
        F.col("Nome da Rua").alias("nome_rua"),
        F.col("Numero Rua").alias("numero_rua"),
        F.col("Complemento").alias("complemento"),
        F.col("Bairro").alias("bairro"),
        F.col("Cep").alias("cep"),
        F.col("Produto").alias("produto"),
        F.col("Data da Coleta").alias("data_coleta"),
        F.col("Valor de Venda").alias("valor_venda"),
        F.col("Valor de Compra").alias("valor_compra"),
        F.col("Unidade de Medida").alias("unidade_medida"),
        F.col("Bandeira").alias("bandeira"),
        # Referencia do arquivo de leitura
        F.col("_metadata.file_name").alias("arquivo_origem"),
        # Timestamp
        F.current_timestamp().alias("data_hora_ingestao")
    )
    .dropDuplicates(["cnpj_revenda", "produto", "data_coleta"])
)

# Armazena dados de precos na tabela delta Bronze
mergeToDelta(
    precosDf,
    tables["PRECOS_REVENDA"],
    """
        target.cnpj_revenda <=> source.cnpj_revenda
        AND target.produto <=> source.produto
        AND target.data_coleta <=> source.data_coleta
    """
)

# Preview
display(precosDf.limit(10))

##### Persistencia dos dados de preços em Parquet

<p>Foi escolhida a persistência dos dados de preços em parquet pois ele constitui o maior conjunto de dados do atual projeto, portanto, dentro do contexto, é a tabela em que há o maior ganho de performance em ser salva desta maneira.</p>
<p>Por ser um formato colunar, o parquet permite que sejam lidas apenas as colunas de interesse, tornando consultas e processamentos mais eficientes.</p>

In [0]:
# Escrita em Parquet
(
    precosDf.write
    .mode("overwrite")
    .format("parquet")
    .save(str(paths["PRECOS_PARQUET"]))
)

# Leitura em Parquet
precosParquetDf = spark.read.parquet(str(paths["PRECOS_PARQUET"]))
print(f"Registros no Parquet: {precosParquetDf.count():,}")

##### Vendas municipais

In [0]:
# Ler dados .csv com spark
vendasRawDf = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(str(Path(paths["VENDAS"] / "*.csv")))
)

# Seleciona e renomeia as colunas a fim de compatibilizar o schema da tabela Bronze 'vendas_municipio'
vendasDf = (
    vendasRawDf
    .select(
        F.col("ANO").alias("ano_referencia"),
        F.col("GRANDE REGIÃO").alias("regiao"),
        F.col("UF").alias("uf"),
        F.col("PRODUTO").alias("produto"),
        F.col("CÓDIGO IBGE").alias("codigo_ibge"),
        F.col("MUNICÍPIO").alias("municipio"),
        F.col("VENDAS").alias("volume_vendido"),
        # Referencia do arquivo de leitura
        F.col("_metadata.file_name").alias("arquivo_origem"),
        # Timestamp
        F.current_timestamp().alias("data_hora_ingestao")
    )
    # Evita dados duplicados; Mesmo ano &  código_municipio & produto negociado
    .dropDuplicates(["ano_referencia", "codigo_ibge", "produto"])
)

mergeToDelta(
    vendasDf,
    tables["VENDAS_MUNICIPIO"],
    """
        target.ano_referencia <=> source.ano_referencia
        AND target.codigo_ibge <=> source.codigo_ibge
        AND target.produto <=> source.produto
    """
)

# Preview
display(vendasDf.limit(10))

##### Municipios IBGE

In [0]:
# Lê arquivo .json com spark
municipiosRawDf = (spark.read.option("multiline", True).json(str(Path(paths["IBGE"] / "municipios.json"))))

# Seleciona e renomeia as colunas a fim de compatibilizar o schema da tabela Bronze 'municipios_ibge'
municipiosDf = (
    municipiosRawDf
    .select(
        F.col("id").alias("codigo_ibge"),
        F.col("nome").alias("municipio"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.UF.id").alias("codigo_uf"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.UF.sigla").alias("uf"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.UF.nome").alias("nome_uf"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.UF.regiao.id").alias("codigo_regiao"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.UF.regiao.sigla").alias("sigla_regiao"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.UF.regiao.nome").alias("nome_regiao"),
        F.col("`regiao-imediata`.id").alias("codigo_regiao_imediata"),
        F.col("`regiao-imediata`.nome").alias("regiao_imediata"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.id").alias("codigo_regiao_intermediaria"),
        F.col("`regiao-imediata`.`regiao-intermediaria`.nome").alias("regiao_intermediaria"),
        F.lit("municipios.json").alias("arquivo_origem"),
        # Timestamp
        F.current_timestamp().alias("data_hora_ingestao")
    )
    # Evita possuirem municipios duplicados com o mesmo código
    .dropDuplicates(["codigo_ibge"])
)

# Armazena dados de municipios na tabela delta Bronze
mergeToDelta(municipiosDf, tables["MUNICIPIOS_IBGE"], "target.codigo_ibge = source.codigo_ibge")

# Preview
display(municipiosDf.limit(10))

##### Dados estaduais do IBGE

In [0]:
# Leitura dos arquivos .json
# - Arquivo de população por estado
statePopulationRawDf = (
    spark.read
    .option("multiline", True)
    .json(str(Path(paths["IBGE"]/ "populacao-estados-2024.json")))
)
# Arquivo de área por estado
stateAreaRawDf = (
    spark.read
    .option("multiline", True)
    .json(str(Path(paths["IBGE"] / "area-estados-2022.json")))
)

# Seleciona e renomeia as colunas a fim de compatibilizar o schema da tabela Bronze 'estados_ibge'
statePopulationDf = (
    statePopulationRawDf
    .select(
        F.col("D1C").alias("codigo_uf"),
        F.col("D3C").alias("ano_populacao"),
        F.col("V").alias("populacao")
    )
    .filter(
        F.col("codigo_uf")
        .cast("long")
        .isNotNull()
    )
)

# Seleciona e renomeia as colunas a fim de compatibilizar o schema da tabela Bronze 'estados_ibge'
stateAreaDf = (
    stateAreaRawDf
    .select(
        F.col("D1C").alias("codigo_uf"),
        F.col("D3C").alias("ano_area"),
        F.col("V").alias("area_km2")
    )
    .filter(
        F.col("codigo_uf")
        .cast("long")
        .isNotNull()
    )
)

##### Criação de dataframe com informações estaduais

In [0]:
# Seleção dos codigos uf por estado
ufsDf = (
    municipiosDf
    .select(
        F.col("codigo_uf")
        .cast("string")
        .alias("codigo_uf"),

        F.col("uf"),
        F.col("nome_uf"),
    )
    .dropDuplicates(["codigo_uf"])
)

# Merge dos dados de população e área por estado
estadosDf = (
    ufsDf
    .join(
        statePopulationDf,
        on="codigo_uf",
        how="inner"
    )
    .join(
        stateAreaDf,
        on="codigo_uf",
        how="inner"
    )
    .select(
        "codigo_uf",
        "uf",
        "nome_uf",
        "ano_populacao",
        "populacao",
        "ano_area",
        "area_km2",
        # Referencia do arquivo de população
        F.lit("populacao-estados-2024.json")
        .alias("arquivo_populacao"),
        # Referencia do arquivo de área
        F.lit("area-estados-2022.json")
        .alias("arquivo_area"),
        # Timestamp
        F.current_timestamp().alias("data_hora_ingestao")
    )
)

# Armazena dados de estados na tabela delta 'estados_ibge'
mergeToDelta(estadosDf, tables["ESTADOS_IBGE"], "target.codigo_uf = source.codigo_uf")

# Preview
display(    estadosDf.orderBy("codigo_uf"))